# Gold: hospital_daily_summary

Builds `adb_amnuay.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from `silver.lab_results`, enriched with hospital name and calendar attributes, plus day-over-day test volume trend via LAG(). Full rebuild every run, per STTM `sttm_silver_to_gold.xlsx`.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_amnuay.gold.hospital_daily_summary
COMMENT 'Gold hospital daily summary: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend. Source: STTM silver to gold.'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                 AS summary_date,
    COUNT(*)                                  AS total_tests,
    COUNT(DISTINCT patient_id)                AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                 AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')      AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')      AS critical_test_count
  FROM adb_amnuay.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(w.hospital_id AS INT)                                                   AS hospital_id,
  CAST(h.hospital_name AS STRING)                                              AS hospital_name,
  CAST(w.summary_date AS DATE)                                                 AS summary_date,
  CAST(c.day_name AS STRING)                                                   AS day_name,
  CAST(c.is_weekend AS BOOLEAN)                                                AS is_weekend,
  CAST(w.total_tests AS INT)                                                   AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                        AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                           AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                           AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                           AS critical_test_count,
  CAST(try_divide(w.abnormal_test_count, w.total_tests) * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(try_divide(w.critical_test_count, w.total_tests) * 100 AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                          AS prev_day_total_tests,
  CAST(
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE (w.total_tests - w.prev_day_total_tests) / w.prev_day_total_tests * 100
    END AS DECIMAL(6,2))                                                       AS test_volume_change_pct
FROM with_lag w
LEFT JOIN adb_amnuay.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_amnuay.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)

In [0]:
# Return the number of records in the rebuilt gold table to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("adb_amnuay.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))